# 08 — The onset specialist (Phase 4)

The model that actually forecasts.

### The problem it exists to solve

Notebook 07 trained on every row. Two thirds of the positive rows at the 1-hour
horizon are roads **already under water**, where the correct answer is sitting in
`fl_depth_now`. Any model optimising one number will spend itself there.

This one is trained **only on rows where the road is currently below the tier**.
It never sees an already-flooded row, so it cannot win by repeating the current
reading. Every row it is scored on is a road that is dry right now and may not
stay that way.

That is the entire job of a flood *warning* system.

### Three things done differently here

**It gets the GFS forecast; the general model does not.** Phase 3 measured this
on all five forecast years: adding `rain_fcst_*` improves onset PR-AUC by 24%
and *degrades* all-rows PR-AUC by 24%. It helps exactly where forecasting is
needed. Fold 1 still trains without it — GFS starts 23 March 2021, and an
imputed forecast is an invented one.

**Negatives are sampled at 25%, not 5%.** Onset rows are already the rare, hard
subset; thinning them further discards the signal the specialist exists to find.

**It is scored on the full population, but may only alert on dry rows.** An event
is defined by the water arriving — and those are precisely the rows the onset
filter removes. Scoring the specialist on its own filtered frame produces zero
events and an event POD of 0.0, which looks like total failure and is a
measurement artefact. The model scores everything; it is simply not permitted to
raise an alert where it was never trained.

### What you need

Notebooks 05, 06 and 07. About 20 minutes — onset models train on more rows.

## Setup

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

from bkkflood.config import load_config
from bkkflood.rawio import connect
from bkkflood.evaluate import folds
from bkkflood.models import run_fold, gain_importance

CFG = load_config()
FOLDS = folds()
TIERS = sorted(int(t) for t in CFG["flood_event"]["tiers_cm"].values())
HORIZONS = CFG["horizons_hours"]
BASE = pd.read_parquet("data/features/baseline_results.parquet")
con = connect()
print("folds:", [f["test"] for f in FOLDS], "| tiers:", TIERS, "| horizons:", HORIZONS)
GEN = pd.read_parquet("data/features/model_general_results.parquet")
print("general-model results loaded:", GEN.shape)

folds: [2022, 2023, 2024, 2025] | tiers: [5, 15, 30] | horizons: [1, 3, 6]
general-model results loaded: (36, 29)


## 1. Train the specialists

In [2]:
results = []
t0 = time.time()
for fold in FOLDS:
    for tier in TIERS:
        for h in HORIZONS:
            try:
                row, booster, meta = run_fold(fold, tier, h, "onset", con=con)
                results.append(row)
            except Exception as e:
                print(f"  SKIP tier={tier} h={h} fold={fold['test']}: {type(e).__name__}: {e}")
    print(f"fold tested on {fold['test']} done  ({time.time()-t0:.0f}s)")

ONSET = pd.concat(results, ignore_index=True)
ONSET.to_parquet("data/features/model_onset_results.parquet", index=False)
print("\n", ONSET.shape, "-> data/features/model_onset_results.parquet")

fold tested on 2022 done  (407s)
fold tested on 2023 done  (779s)
fold tested on 2024 done  (1295s)
fold tested on 2025 done  (2276s)

 (36, 29) -> data/features/model_onset_results.parquet


## 2. The head-to-head that decides Phase 4

Four contenders on the same test years, at 15 cm / 1 hour.

**Read the columns in this order:** `recall_onset` first, then `event_pod`, then
`median_lead_minutes`. PR-AUC and overall recall are last, and on this table they
are close to decoration — a model can win both by describing water that has
already arrived.

In [3]:
cols = ["pr_auc", "precision", "recall", "recall_onset", "f2", "event_pod",
        "median_lead_minutes"]

def table(tier, h):
    rows = {}
    for name, df in [("lightgbm_onset", ONSET), ("lightgbm_general", GEN)]:
        d = df[(df.tier_cm == tier) & (df.horizon_h == h)]
        rows[name] = d[cols].mean()
    b = BASE[(BASE.tier_cm == tier) & (BASE.horizon_h == h)]
    for name, d in b.groupby("baseline"):
        rows[name] = d.reindex(columns=cols).mean()
    return pd.DataFrame(rows).T.round(4)

print("15 cm, 1 hour ahead\n")
print(table(15, 1).to_string())
print()
print("(baselines have no event_pod / lead columns here — notebook 06 computes")
print(" those separately; see docs/reports/baselines.md)")

15 cm, 1 hour ahead

                  pr_auc  precision  recall  recall_onset      f2  event_pod  median_lead_minutes
lightgbm_onset    0.1029     0.1626  0.2286        0.2286  0.2112     0.5988                 15.0
lightgbm_general  0.5656     0.6305  0.5581        0.1899  0.5708     0.4870                 15.0
always_negative   0.0004     0.0000  0.0000        0.0000  0.0000        NaN                  NaN
climatology       0.0009     0.0019  0.1037        0.0952  0.0085        NaN                  NaN
persistence       0.5529     0.6175  0.5443        0.1635  0.5541        NaN                  NaN
rain_rule         0.1035     0.1066  0.3154        0.2180  0.2185        NaN                  NaN

(baselines have no event_pod / lead columns here — notebook 06 computes
 those separately; see docs/reports/baselines.md)


In [4]:
o = ONSET[(ONSET.tier_cm == 15) & (ONSET.horizon_h == 1)]
g = GEN[(GEN.tier_cm == 15) & (GEN.horizon_h == 1)]
p = BASE[(BASE.tier_cm == 15) & (BASE.horizon_h == 1) & (BASE.baseline == "persistence")]
r = BASE[(BASE.tier_cm == 15) & (BASE.horizon_h == 1) & (BASE.baseline == "rain_rule")]

print("ONSET RECALL — recall on roads that were dry when the forecast was made\n")
for name, v in [("onset specialist", o.recall_onset.mean()),
                ("general model   ", g.recall_onset.mean()),
                ("rain rule       ", r.recall_onset.mean()),
                ("persistence     ", p.recall_onset.mean())]:
    print(f"  {name}  {v:.1%}")

print("\nEVENT POD — floods flagged before the water arrived\n")
for name, v in [("onset specialist", o.event_pod.mean()),
                ("general model   ", g.event_pod.mean())]:
    print(f"  {name}  {v:.1%}   median lead "
          f"{(o if 'onset' in name else g).median_lead_minutes.mean():.0f} min")

print()
if o.recall_onset.mean() > max(g.recall_onset.mean(), p.recall_onset.mean(),
                               r.recall_onset.mean()):
    print("The specialist wins on the metric it was built for.")
else:
    print("The specialist does NOT lead on onset recall. Do not deploy it on the")
    print("strength of its other numbers — report this plainly instead.")

ONSET RECALL — recall on roads that were dry when the forecast was made

  onset specialist  22.9%
  general model     19.0%
  rain rule         21.8%
  persistence       16.4%

EVENT POD — floods flagged before the water arrived

  onset specialist  59.9%   median lead 15 min
  general model     48.7%   median lead 15 min

The specialist wins on the metric it was built for.


## 3. What the specialist learned

Compare this against notebook 07's importance table. The interesting question is
whether removing the already-flooded rows moved the model away from reading the
depth sensor and toward rainfall, terrain and canal state.

In [5]:
row, booster, meta = run_fold(FOLDS[-1], 15, 1, "onset", con=con,
                              save_as="onset_t15_h1_final")
imp = gain_importance(booster, 15)
print(f"most recent fold (test {FOLDS[-1]['test']}), 15 cm / 1 h")
print(f"features: {meta['n_features']}, GFS forecast included: {meta['gfs_included']}\n")
print(imp.round(4).to_string(index=False))

groups = {"own depth history": "fl_", "rainfall": "rain_", "canal": ("water_", "flow_"),
          "terrain": "terr_", "calendar / tide": ("cal_", "tide_")}
print("\nshare of gain by feature family:")
for label, pref in groups.items():
    pref = pref if isinstance(pref, tuple) else (pref,)
    share = imp[imp.feature.str.startswith(pref)].gain_share.sum()
    print(f"  {label:20s} {share:.1%}")

most recent fold (test 2025), 15 cm / 1 h
features: 50, GFS forecast included: True

            feature         gain  splits  gain_share
         fl_rise_1h 1478193.2959     132      0.6667
      fl_rise_15min  231004.3305     208      0.1042
         fl_mean_1h   79774.3097     182      0.0360
    fl_depth_lag_1h   35352.1913      44      0.0159
          fl_std_3h   34823.3497     108      0.0157
    rain_rf1hr_mean   33241.0138      48      0.0150
       fl_depth_now   23485.0684     135      0.0106
       rain_fcst_6h   17234.4922     383      0.0078
        cal_doy_cos   17061.8821     549      0.0077
   tide_spring_neap   16924.8774     645      0.0076
fl_hours_since_15cm   15966.2787     475      0.0072
          fl_max_3h   14978.9236      61      0.0068
        cal_doy_sin   13141.9530     501      0.0059
 water_rising_share   12809.9785     295      0.0058
 fl_hours_since_5cm   11892.5293     430      0.0054

share of gain by feature family:
  own depth history    86.8%
  ra

## 4. Horizon and tier

Onset recall should fall as the horizon lengthens — six hours ahead is a harder
question than one hour. If it does not, something is wrong.

At 30 cm there are roughly 43 positive rows in a year. Whatever appears there is
noise, and it is printed so that nobody quotes it as a result.

In [6]:
grid = ONSET.groupby(["tier_cm", "horizon_h"])[
    ["base_rate", "pr_auc", "recall_onset", "event_pod",
     "median_lead_minutes", "positives"]].mean().round(4)
print(grid.to_string())

print("\npositives available per test year, by tier:")
print(ONSET.groupby("tier_cm").positives.mean().round(0).to_string())
print("\nA tier with a few dozen positives cannot support a trained model. Report")
print("it as a data limitation, never as a score.")

                   base_rate  pr_auc  recall_onset  event_pod  median_lead_minutes  positives
tier_cm horizon_h                                                                            
5       1             0.0006  0.0375        0.1665     0.3465               22.500    2186.75
        3             0.0018  0.0193        0.1409     0.4989               30.000    6232.50
        6             0.0036  0.0198        0.1943     0.5527               93.750   12180.25
15      1             0.0002  0.1029        0.2286     0.5988               15.000     624.50
        3             0.0005  0.0341        0.0976     0.6361               15.000    1777.00
        6             0.0010  0.0130        0.0904     0.7092               16.875    3484.50
30      1             0.0000  0.1668        0.3100     0.7246               15.000     107.25
        3             0.0001  0.0669        0.1531     0.7960               18.750     286.00
        6             0.0002  0.0292        0.0995     0.878

## 5. Where it fails

The false negatives are the floods that would have gone unwarned. Worth looking
at directly rather than as a rate — Phase 5 builds this out properly, but a first
look belongs next to the model that produced them.

In [7]:
test = FOLDS[-1]["test"]
from bkkflood.models import score_year
from bkkflood.evaluate import best_threshold

val = score_year(booster, FOLDS[-1]["val"], meta["features"], 15, 1,
                 kind="onset", negative_rate=0.25, con=con)
thr = best_threshold(val["y_ge15_1h"].fillna(False).to_numpy(bool),
                     val["score"].to_numpy(), metric="f2")["threshold"]
sc = score_year(booster, test, meta["features"], 15, 1, kind="onset",
                negative_rate=0.25, con=con, full_population=True)
sc["pred"] = (sc["score"] >= thr) & sc["onset_alertable"]
sc["ytrue"] = sc["y_ge15_1h"].fillna(False)

fn = sc[sc.ytrue & ~sc.pred]
print(f"missed onset rows in {test}: {len(fn):,}\n")
print("stations with the most misses:")
print(fn.station_code.value_counts().head(10).to_string())
print("\nmisses by month (1 = January):")
print(fn.ts.dt.month.value_counts().sort_index().to_string())

missed onset rows in 2025: 924

stations with the most misses:
station_code
FL.SMI.01    73
FL.BNA.04    48
FL.BKM.01    46
FL.CTC.04    44
FL.CTC.07    42
FL.DDG.02    41
FL.CTC.03    33
FL.SLG.03    32
FL.STN.03    32
FL.LSI.02    30

misses by month (1 = January):
ts
2       3
4      11
5     253
6       9
7       9
8      20
9     136
10     60
11    419
12      4


In [8]:
out = Path("docs/reports/model_onset.md")
with out.open("w") as f:
    f.write("# LightGBM — onset specialist\n\n")
    f.write("Generated by `notebooks/08_train_onset.ipynb`. Trained only on rows where\n")
    f.write("the road was below the tier, so it cannot score by repeating the current\n")
    f.write("sensor reading.\n\n")
    f.write("## Head to head: 15 cm, 1 hour\n\n")
    f.write(table(15, 1).to_markdown())
    f.write("\n\n## All tiers and horizons\n\n")
    f.write(grid.to_markdown())
    f.write("\n\n## Feature families used (most recent fold, 15 cm / 1 h)\n\n")
    for label, pref in groups.items():
        pr = pref if isinstance(pref, tuple) else (pref,)
        f.write(f"- {label}: {imp[imp.feature.str.startswith(pr)].gain_share.sum():.1%}\n")
    f.write("\n## Promotion rule\n\n")
    f.write("Deploy the specialist only if it leads on **onset recall and event POD**\n")
    f.write("at the 15 cm tier. Leading on overall recall or PR-AUC proves only that\n")
    f.write("it has also learned to repeat the current reading.\n")
print("wrote", out)

wrote docs/reports/model_onset.md


## What Phase 4 has so far

| Model | Question it answers |
|---|---|
| general | will this station be at the tier within h hours? |
| **onset** | **this road is dry — will it flood within h hours?** |

**Next:** `09_train_depth_quantiles.ipynb`. A binary alert says *whether*.
An operations room also needs *how deep*, and the p95 depth is what config maps
to CAP severity levels.